In [2]:
import json
import pandas as pd
from uniguide.config import get_path
from uniguide.data.splits import apply_cold_start, check_no_leakage

p = get_path("processed")
train, val, test = (pd.read_parquet(p / f"{n}.parquet") for n in ["train", "val", "test"])
cold_users = json.loads((p / "cold_users.json").read_text())
cold_courses = json.loads((p / "cold_courses.json").read_text())

check_no_leakage(train, val, test)          # explicit check required by the project
pd.read_csv(get_path("results") / "tables" / "splits_summary.csv")

,set,n_rows,n_users,n_courses
0,train,81530,14756,380
1,val,16983,14756,378
2,test,24629,14756,380
3,cold_course_test,15434,9514,42


In [3]:
# Cold-start users: history size per group after truncation
truncated = apply_cold_start(pd.concat([train, val]), cold_users)
for size, users in cold_users.items():
    kept = truncated[truncated["user_id"].isin(users)].groupby("user_id").size()
    print(f"group {size}: {len(users)} users, kept interactions "
          f"min={kept.min() if len(kept) else 0}, max={kept.max() if len(kept) else 0}")

group 0: 445 users, kept interactions min=0, max=0
group 1: 445 users, kept interactions min=1, max=1
group 3: 445 users, kept interactions min=3, max=3
group 5: 445 users, kept interactions min=5, max=5


In [4]:
# Cold-start courses never appear in train/val/test
assert not set(cold_courses) & set(pd.concat([train, val, test])["course_id"])
courses = pd.read_parquet(p / "courses.parquet")
courses[courses["course_id"].isin(cold_courses)][["name", "text", "n_reviews"]].head()

,name,text,n_reviews
2,The Science of Well-Being,science yale university,0
5,Introduction to Psychology,introduction psychology yale university,0
6,Programming for Everybody (Getting Started wit...,programming everybody getting started python u...,0
59,Understanding the Brain: The Neurobiology of E...,understanding brain neurobiology everyday life...,0
95,Learning to Teach Online,learning teach online unsw sydney university n...,0
